# Deep Agents on GAIA → INIF

End-to-end tutorial that runs an [Inspect AI **Deep Agent**](https://inspect.aisi.org.uk/deepagent.html) (a `react` loop equipped with web search, a Python sandbox, and a structured `think` scratchpad) over a small slice of the [GAIA benchmark](https://huggingface.co/datasets/gaia-benchmark/GAIA), then loads the resulting eval log into INIF for trace-level analysis.

**Model:** [`moonshotai/Kimi-K2.6`](https://huggingface.co/moonshotai/Kimi-K2.6) via Together AI — the same family of long-context thinking MoE used in `inspect_gpqa.ipynb`, just one minor revision newer.

**Steps:**
1. Load a few GAIA records from HuggingFace and turn them into Inspect `Sample`s.
2. Build a Deep-Agent `Task` with `react(...)` + `web_search`, `python`, `think`, and `bash` tools.
3. Run the eval against Kimi K2.6 on Together; save the `.eval` log under `devtest/logs/`.
4. Convert the log into an `InifDocument` with the real Kimi tokenizer.
5. Inspect the converted structure — tool-call rendering, generated tokens, chat roles.
6. Tag tool-call boundaries and split correct vs. incorrect runs.
7. Render the trace inline with `doc.show()` and persist the enriched archive.

> **Cost guard.** The defaults below run **3 GAIA Level-1 questions** with `max_steps=8` and `max_tokens=8000` per turn. That keeps the smoke run cheap; bump `LIMIT`, `MAX_STEPS`, or `MAX_TOKENS` once you've validated the pipeline.

## 0. Setup

GAIA is a gated dataset on the HuggingFace Hub: you must first accept the licence at [https://huggingface.co/datasets/gaia-benchmark/GAIA](https://huggingface.co/datasets/gaia-benchmark/GAIA), then `huggingface-cli login` (or set `HF_TOKEN`) so `datasets.load_dataset` can pull it.

The cell below loads keys from `devtest/.env`. Required:

- `TOGETHER_API_KEY` — the model call.

Optional (enables `web_search`; without it, the agent will only have `python` / `bash` / `think`):

- `TAVILY_API_KEY` — sign up at [tavily.com](https://tavily.com). Inspect's `web_search()` Tavily provider hits `/search`, which returns titles + URLs + page snippets the agent can quote — exactly what GAIA's "use the value on this Wikipedia page" style of questions need. We turn on `include_raw_content` and `search_depth="advanced"` so the model sees the page text rather than a one-line summary.

In [ ]:
%%capture
!uv pip install "inif[inspect]" inspect-ai datasets transformers jinja2

In [ ]:
import os
from pathlib import Path

DEVTEST = Path.cwd()
ENV_PATH = DEVTEST / ".env"
assert ENV_PATH.exists(), f"Expected {ENV_PATH} with TOGETHER_API_KEY=..."
for line in ENV_PATH.read_text().splitlines():
    line = line.strip()
    if not line or line.startswith("#") or "=" not in line:
        continue
    key, value = line.split("=", 1)
    os.environ.setdefault(key.strip(), value.strip())

assert os.environ.get("TOGETHER_API_KEY"), "TOGETHER_API_KEY missing from .env"
print("Together credentials loaded.")

# Inspect's `web_search()` raises "No valid provider found" if you ask for a
# provider whose key isn't set, so enable it only when TAVILY_API_KEY is present.
SEARCH_PROVIDER = "tavily" if os.environ.get("TAVILY_API_KEY") else None

if SEARCH_PROVIDER:
    print(f"Web-search provider: {SEARCH_PROVIDER}")
else:
    print(
        "TAVILY_API_KEY not found — agent will run with python / bash / think only.\n"
        "Add TAVILY_API_KEY to devtest/.env to enable web search."
    )

## 1. Build an Inspect dataset from GAIA

GAIA ships three difficulty splits (Level 1 / 2 / 3) under the `2023_all` config; we sample from `validation` so we get gold answers. Each row carries a `Question`, `Final answer`, and an annotator metadata block describing the tools the human solver used — handy for cross-checking what the agent reaches for.

We promote each row to an `inspect_ai.Sample` with the question as `input`, `Final answer` as `target`, and the annotator metadata stashed in `metadata` for later analysis.

In [ ]:
from datasets import load_dataset
from inspect_ai.dataset import MemoryDataset, Sample as InspectSample

LIMIT = 3  # number of GAIA questions to run; raise to scale up
LEVEL = "1"  # GAIA Level 1 questions are the cheapest / fastest to debug

raw = load_dataset("gaia-benchmark/GAIA", "2023_all", split="validation")
level_rows = [row for row in raw if str(row.get("Level", "")) == LEVEL][:LIMIT]


def to_inspect_sample(row: dict) -> InspectSample:
    annotator = row.get("Annotator Metadata") or {}
    return InspectSample(
        id=row["task_id"],
        input=row["Question"],
        target=row["Final answer"],
        metadata={
            "level": row.get("Level"),
            "annotator_steps": annotator.get("Steps"),
            "annotator_tools": annotator.get("Tools"),
            "annotator_num_tools": annotator.get("Number of tools"),
            "file_name": row.get("file_name") or None,
        },
    )


samples = [to_inspect_sample(row) for row in level_rows]
dataset = MemoryDataset(samples=samples, name=f"gaia_level{LEVEL}_smoke")

print(f"Built dataset with {len(dataset)} GAIA Level {LEVEL} samples")
for s in samples:
    preview = (s.input[:90] + "…") if len(s.input) > 90 else s.input
    print(f"  [{s.id}] target={s.target!r}")
    print(f"          {preview}")

## 2. Define the Deep Agent

Following [the Inspect Deep Agent recipe](https://inspect.aisi.org.uk/deepagent.html), we wire a `react(...)` loop equipped with:

| Tool | Why GAIA needs it | Requires |
| --- | --- | --- |
| `custom_web_search` | most GAIA questions need ground-truth lookups; built below — Tavily search → fetch top URLs → Kimi K2.6 summariser, so page-specific values survive Kimi's chat template (Inspect's bundled `web_search` only forwards Tavily's one-line `answer`). | `TAVILY_API_KEY` + `TOGETHER_API_KEY` |
| `python` | numeric reasoning, light data manipulation | — |
| `bash` | downloading attachments, peeking at files | — |
| `think` | structured scratchpad — emitted as a separate tool call so we can tag it later in INIF | — |

`react` exits when the model calls the built-in `submit` tool with its final answer. The system prompt nudges the model to think → tool-use → answer in the GAIA exact-match style. If `TAVILY_API_KEY` is not configured, the agent still runs but is restricted to whatever it can derive from `python` / `bash` reasoning alone — expect lower accuracy on knowledge-heavy questions.

### 2a. Custom `web_search` tool (Tavily → fetch → Kimi summary)

Inspect's bundled `web_search()` Tavily provider only renders Tavily's synthesised one-line `answer` into the tool message — Kimi K2.6's `chat_template.jinja` only emits `content[i].text` and ignores the `citations` array, so the page snippets are stripped before they reach the model. That's why the agent kept asking "what does the Wikipedia page actually say?" without ever seeing the page text.

The tool below replaces it:

1. Hit Tavily `/search` with the agent's query.
2. Pull the top citation URLs out of the response.
3. Fetch each URL (HTML → readable text via BeautifulSoup, capped at a few KB).
4. Call Kimi K2.6 on Together with the concatenated pages and the original query, asking for a focused summary that quotes verbatim where the question pins a specific page.

The summariser-Kimi runs at `temperature=0` for stability and is given a tight prompt that mirrors GAIA's style (quote exact values, prefer the source the user asked for). The summary is the *only* string returned by the tool, so it's what the agent sees in its rendered tool message.

In [ ]:
%%capture
!uv pip install tavily-python beautifulsoup4 httpx together

In [ ]:
from __future__ import annotations

import asyncio

import httpx
from bs4 import BeautifulSoup
from tavily import AsyncTavilyClient
from together import AsyncTogether

from inspect_ai.tool import Tool, ToolError, tool

# --- Tunables --------------------------------------------------------------
SEARCH_MAX_RESULTS = 5  # Tavily citations to consider
FETCH_MAX_URLS = 3  # how many of the citations we actually fetch
FETCH_TIMEOUT = 15.0  # per-URL fetch timeout (s)
FETCH_CHAR_CAP = 12_000  # truncate per-page text before summarisation
SUMMARY_MAX_TOKENS = 700  # Kimi summariser output cap
SUMMARISER_MODEL = "moonshotai/Kimi-K2.6"

# Reusable async clients — instantiated once per kernel.
_tavily = AsyncTavilyClient(api_key=os.environ["TAVILY_API_KEY"])
_together = AsyncTogether(api_key=os.environ["TOGETHER_API_KEY"])
_http = httpx.AsyncClient(
    timeout=FETCH_TIMEOUT,
    follow_redirects=True,
    headers={"User-Agent": "inif-deepagent/1.0 (+research)"},
)


async def _fetch_url(url: str) -> str:
    """Fetch ``url`` and return readable text (BeautifulSoup, no scripts/style)."""
    resp = await _http.get(url)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    for el in soup(["script", "style", "noscript", "nav", "footer", "header"]):
        el.decompose()
    text = soup.get_text(separator="\n")
    # collapse runs of blank lines so the cap covers more real content
    lines = [ln.strip() for ln in text.splitlines()]
    text = "\n".join(ln for ln in lines if ln)
    return text[:FETCH_CHAR_CAP]


SUMMARISER_PROMPT = """\
You are a research summariser for an autonomous agent answering a benchmark question.

Question / search query: {query}

Below are the contents of the top web pages returned for this query. Your job is to
extract ONLY the facts that help answer the question.

Rules:
- If the question pins a specific source (e.g. "the Wikipedia page for X"), prefer
  that source and quote the relevant value verbatim.
- Include exact numbers, table rows, and dates when they appear in the text.
- Cite each fact with the source URL inline as [URL].
- Be concise (≤ 400 words). Do not speculate or add knowledge that is not in the
  fetched pages.
- If none of the pages answer the question, say so explicitly and list which
  sources you checked.

Pages:
{pages}
"""


@tool
def custom_web_search() -> Tool:
    """Web search tool: Tavily → fetch top URLs → Kimi summary.

    The model sees the Kimi-generated summary verbatim in the tool message, so
    page-specific values survive the chat-template rendering (unlike
    ``inspect_ai.tool.web_search`` whose ``citations`` field is dropped by
    Kimi's chat_template).
    """

    async def execute(query: str) -> str:
        """Search the web and return a focused summary of the top pages.

        Args:
            query: Natural-language search query. Include a target domain
                (e.g. ``site:wikipedia.org``) when the question pins a source.
        """
        if not query or not query.strip():
            raise ToolError("query must be a non-empty string")

        # 1. Tavily search.
        tavily_resp = await _tavily.search(
            query=query,
            search_depth="advanced",
            max_results=SEARCH_MAX_RESULTS,
            include_raw_content=False,
        )
        results = tavily_resp.get("results") or []
        if not results:
            return f"No web results for query: {query!r}"

        # 2. Pick top URLs.
        urls = [r["url"] for r in results[:FETCH_MAX_URLS] if r.get("url")]

        # 3. Fetch concurrently. Drop fetches that fail; keep going if at
        #    least one URL succeeds so the summariser has something to chew on.
        async def _safe_fetch(url: str) -> tuple[str, str | None]:
            try:
                return url, await _fetch_url(url)
            except Exception as exc:
                return url, f"<<fetch error: {type(exc).__name__}: {exc}>>"

        fetched = await asyncio.gather(*[_safe_fetch(u) for u in urls])

        pages_block = "\n\n".join(
            f"### Source {i + 1}: {url}\n{body}"
            for i, (url, body) in enumerate(fetched)
            if body is not None
        )
        if not pages_block.strip():
            return f"All page fetches failed for query: {query!r}\nURLs tried: {urls}"

        # 4. Summarise with Kimi K2.6 on Together.
        completion = await _together.chat.completions.create(
            model=SUMMARISER_MODEL,
            temperature=0.0,
            max_tokens=SUMMARY_MAX_TOKENS,
            messages=[
                {
                    "role": "user",
                    "content": SUMMARISER_PROMPT.format(query=query, pages=pages_block),
                }
            ],
        )
        summary = completion.choices[0].message.content or ""
        sources_line = "Sources fetched:\n" + "\n".join(f"- {u}" for u in urls)
        return f"{summary.strip()}\n\n{sources_line}"

    return execute


print("custom_web_search tool defined.")

In [ ]:
from inspect_ai import Task
from inspect_ai.agent import react
from inspect_ai.scorer import match
from inspect_ai.tool import bash, python, think

MAX_STEPS = 8  # max react iterations per sample (cap on tool-use loop)
MAX_TOKENS = 8000  # per-turn output cap; GAIA answers are short, reasoning isn't

DEEP_AGENT_PROMPT = """\
You are a deep research agent answering questions from the GAIA benchmark.
Plan briefly with the `think` tool, then use the available tools (custom_web_search
when you need facts from the web, otherwise python / bash) to gather evidence.
The custom_web_search tool returns a focused summary of the top pages for your
query — include `site:wikipedia.org` (or another domain) when the question
pins a specific source. When you are confident, call `submit` with ONLY the
final answer — no prose, no units unless explicitly requested by the question,
and no surrounding quotes.
"""

agent_tools = [python(timeout=60), bash(timeout=60), think()]
if SEARCH_PROVIDER == "tavily":
    # Replace Inspect's bundled web_search with our Tavily → fetch → Kimi
    # summariser. Inspect's version only renders Tavily's one-line `answer` into
    # the tool message (Kimi's chat_template drops the `citations` array), so
    # page-specific values never reach the agent. Our tool stitches the fetched
    # page text into the summary string, which is what the chat template emits.
    agent_tools.insert(0, custom_web_search())

deep_agent = react(
    name="gaia_deep_agent",
    description="Deep agent for GAIA — custom_web_search + python + think + bash",
    prompt=DEEP_AGENT_PROMPT,
    tools=agent_tools,
)

task = Task(
    dataset=dataset,
    solver=deep_agent,
    scorer=match(location="any"),
    name=f"gaia_level{LEVEL}_deepagent",
    # python() and bash() require a sandbox. "local" runs them on the host
    # without containerisation — fine for a notebook smoke run; switch to
    # "docker" (and `docker compose`) for untrusted code or production.
    sandbox="local",
)
print(f"Task ready: {task.name}")
print(f"Tools wired: {[getattr(t, '__name__', type(t).__name__) for t in agent_tools]}")

## 3. Run the eval

`max_messages` sits one above `MAX_STEPS * 2` so we have headroom for the auto-submitted final message. Logs land under `devtest/logs/` so this notebook stays self-contained.

> **Heads up.** Even with `LIMIT=3` and `MAX_STEPS=8` this issues real Together API calls. Comment out the `eval(...)` cell and skip directly to step 4 if you already have a log in `devtest/logs/` you'd rather analyse.

In [ ]:
from inspect_ai import eval
from inspect_ai.model import GenerateConfig

MODEL = "together/moonshotai/Kimi-K2.6"
LOG_DIR = DEVTEST / "logs"
LOG_DIR.mkdir(exist_ok=True)

logs = eval(
    tasks=task,
    model=MODEL,
    log_dir=str(LOG_DIR),
    log_format="eval",
    max_connections=3,
    max_messages=MAX_STEPS * 2 + 2,
    config=GenerateConfig(
        max_tokens=MAX_TOKENS,
        temperature=0.2,
        seed=42,
    ),
)

log = logs[0]
print(f"Wrote: {log.location}")
print(f"Samples: {len(log.samples)}")
for s in log.samples:
    score = next(iter((s.scores or {}).values()), None)
    value = getattr(score, "value", None)
    answer = getattr(score, "answer", None)
    print(f"  [{s.id}] target={s.target!r}  pred={answer!r}  score={value}")

## 4. Convert the eval log to INIF

Two things happen here:

1. **`.eval` → `.eval.json`.** Inspect's binary `.eval` archive is convenient for Inspect's own tooling but opaque to grep / diff / external readers. `write_eval_log(log, ..., format="json")` re-serialises the same `EvalLog` as plain JSON — keeping fields the INIF converter does not (yet) extract, like the model's tool-call format/spec, per-message `tool_choice`, raw provider responses, and Inspect-internal step events. Useful as a reference when the INIF view feels lossy.\n",
2. **`.eval` → `InifDocument`.** `from_eval_file` reads the `.eval` archive, tokenizes every message with the supplied HuggingFace tokenizer via `apply_chat_template`, deduplicates the shared system-prompt prefix into a `Sequence`, and tags chat roles + the `generated` span. We pass the real Kimi K2.6 tokenizer so per-token ids match what the model saw on Together (Together routes through Moonshot's hosted weights but the tokenizer is the public HF one)."

In [ ]:
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path
from inspect_ai.log import read_eval_log, write_eval_log  # noqa: E402
from transformers import AutoTokenizer  # noqa: E402

from inif.converters.inspect_ai import from_eval_file  # noqa: E402

# 1. Side-by-side .eval.json export — keeps every field Inspect tracks (tool-call
#    format / spec, tool_choice, raw provider responses, step events, …) so we
#    have a reference in case INIF's converter drops something we care about.
#eval_path = Path("devtest/logs/2026-04-28T20-18-14-00-00_gaia-level1-deepagent_bcNJfFJHQtaXYyFH2RNTTZ.eval")
eval_path = Path(log.location)
json_log_path = eval_path.with_suffix(".eval.json")
write_eval_log(read_eval_log(str(eval_path)), str(json_log_path), format="json")
print(f"Wrote JSON eval log: {json_log_path}  ({json_log_path.stat().st_size:,} bytes)")

# 2. Convert to INIF.
tokenizer = AutoTokenizer.from_pretrained(
    "moonshotai/Kimi-K2.6", trust_remote_code=True
)

doc = from_eval_file(str(eval_path), tokenizer=tokenizer)

meta = doc.metadata
print(f"\nModel:           {meta.model.name}")
print(f"Task:            {meta.source_eval.task}")
print(f"Inspect version: {meta.source_eval.framework_version}")
print(f"Total time:      {meta.total_time:.1f}s")
print(f"Samples:         {doc.total_samples}")
print(f"Sequences:       {len(doc.sequences)} (shared token runs)")

## 5. Inspect the converted structure

Each `Sample` carries the chat-templated token stream (with the system+tool-spec prefix collapsed into a `Sequence`), the eval `score`, and the gold `target`. Per-message roles already live on `Sample.annotations`; the converter also writes the `generated` annotation across the LAST assistant message.

In [ ]:
from collections import Counter

for sample in doc.samples:
    n_ref = sum(1 for t in sample.tokens if t.is_sequence_ref)
    n_own = sum(1 for t in sample.tokens if not t.is_sequence_ref)
    role_counts = {}
    for ann in sample.annotations:
        if ann.metadata.get("source") == "message_role":
            role_counts[ann.name] = sum(end - start for start, end in ann.ranges)
    score = sample.scores[0] if sample.scores else None
    print(f"Sample [{sample.id}]")
    print(f"  target:    {sample.target!r}")
    if score is not None:
        print(f"  scored:    {score.value}  (answer={score.answer!r})")
    print(f"  tokens:    {n_ref} ref(s) + {n_own} own ({n_ref + n_own} total)")
    print(f"  roles:     {role_counts}")
    print(f"  generated: {len(sample.annotation_positions('generated'))} tokens")
    print()

## 7. Inline visualisation

`InifDocument.show()` renders the whole document as colored HTML inline. With only a handful of samples that's fine; for larger eval logs use `doc.subset(...)` to scope down to one trace at a time before rendering.

In [ ]:
doc.show()

## 8. Persist the enriched document

Save both the plain JSON form (easy to diff and grep) and the indexed `.inif` archive (random-access reads via `read_samples` / `read_info`). The `tool_call`, `think_open`, and chat-role annotations all survive the round trip.

In [ ]:
from inif import InifDocument, read_info, validate


from pathlib import Path

DEVTEST = Path.cwd()

out_json = DEVTEST / "gaia_deepagent.inif.json"
out_inif = DEVTEST / "gaia_deepagent.inif"

doc.save(out_json)
doc.save(out_inif)

validate(doc.to_dict())

info = read_info(out_inif)
reloaded = InifDocument.load(out_inif)
n_tool_round_trip = sum(
    len(s.annotation_positions("tool_call")) for s in reloaded.samples
)
print(f"Saved JSON:    {out_json}")
print(f"Saved indexed: {out_inif}")
print(
    f"Reloaded:      {reloaded.total_samples} samples; "
    f"tool_call annotations preserved: {n_tool_round_trip}"
)
print("\nHeader-only summary (read_info):")
for s in info.samples:
    print(f"  [{s['id']}] n_tokens={s['n_tokens']}  scores={s.get('scores')}")

## 9. Re-run the same eval with Qwen via HF Inference Providers

Same `task`, different model: `Qwen/Qwen3.6-35B-A3B` served through [HF Inference Providers](https://huggingface.co/docs/inference-providers/guides/evaluation-inspect-ai). The `:featherless-ai` suffix pins the route to Featherless (the provider you have enabled on your HF account); without it the router refuses with `model_not_supported` because the auto-route doesn't know to pick Featherless. Only `HF_TOKEN` is needed — no extra provider keys.


In [ ]:
import os

assert os.environ.get("HF_TOKEN"), "HF_TOKEN missing from .env"
print("HF Inference Providers credentials loaded.")

In [ ]:
from inspect_ai import eval
from inspect_ai.model import GenerateConfig

LOG_DIR = DEVTEST / "logs"
LOG_DIR.mkdir(exist_ok=True)

# `hf-inference-providers/<owner>/<model>:<provider>` lets Inspect dispatch
# through the HF router and pin a specific upstream provider. Featherless is
# the provider you have enabled on your HF account; the suffix is required
# here because the bulk auto-route doesn't surface this checkpoint without it
# (the router returns model_not_supported otherwise).
HF_MODEL_ID = "Qwen/Qwen3.6-35B-A3B"
HF_PROVIDER = "featherless-ai"
HF_MODEL = f"hf-inference-providers/{HF_MODEL_ID}:{HF_PROVIDER}"

# Featherless serverless can take well over a minute to emit the first token
# on long reasoning + tool-call generations, so override Inspect's 60s
# default; max_retries lets us recover from transient backend stalls without
# losing the run.
logs_qwen = eval(
    tasks=task,
    model=HF_MODEL,
    log_dir=str(LOG_DIR),
    log_format="eval",
    max_connections=3,
    model_args={"stream": False, "client_timeout": 600},
    max_messages=MAX_STEPS * 2 + 2,
    config=GenerateConfig(
        max_tokens=MAX_TOKENS,
        temperature=0.2,
        seed=42,
        timeout=600,
        max_retries=3,
    ),
)

log_qwen = logs_qwen[0]
print(f"Wrote: {log_qwen.location}")
print(f"Samples: {len(log_qwen.samples)}")
for s in log_qwen.samples:
    score = next(iter((s.scores or {}).values()), None)
    value = getattr(score, "value", None)
    answer = getattr(score, "answer", None)
    print(f"  [{s.id}] target={s.target!r}  pred={answer!r}  score={value}")


In [ ]:
qwen_eval_path = Path(log_qwen.location)
qwen_json_log_path = qwen_eval_path.with_suffix(".eval.json")
write_eval_log(read_eval_log(str(qwen_eval_path)), str(qwen_json_log_path), format="json")
print(f"Wrote JSON eval log: {qwen_json_log_path}  ({qwen_json_log_path.stat().st_size:,} bytes)")

qwen_tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_ID, trust_remote_code=True)
doc_qwen = from_eval_file(str(qwen_eval_path), tokenizer=qwen_tokenizer)

meta = doc_qwen.metadata
print(f"\nModel:           {meta.model.name}")
print(f"Task:            {meta.source_eval.task}")
print(f"Inspect version: {meta.source_eval.framework_version}")
print(f"Total time:      {meta.total_time:.1f}s")
print(f"Samples:         {doc_qwen.total_samples}")
print(f"Sequences:       {len(doc_qwen.sequences)} (shared token runs)")


In [ ]:
doc_qwen.show()


## 12. Persist the Qwen document

Mirror the Kimi save path so both traces sit side by side under `devtest/`.


In [ ]:
qwen_out_json = DEVTEST / "gaia_deepagent_qwen.inif.json"
qwen_out_inif = DEVTEST / "gaia_deepagent_qwen.inif"

doc_qwen.save(qwen_out_json)
doc_qwen.save(qwen_out_inif)

validate(doc_qwen.to_dict())

qwen_info = read_info(qwen_out_inif)
qwen_reloaded = InifDocument.load(qwen_out_inif)
n_tool_round_trip = sum(
    len(s.annotation_positions("tool_call")) for s in qwen_reloaded.samples
)
print(f"Saved JSON:    {qwen_out_json}")
print(f"Saved indexed: {qwen_out_inif}")
print(
    f"Reloaded:      {qwen_reloaded.total_samples} samples; "
    f"tool_call annotations preserved: {n_tool_round_trip}"
)
print("\nHeader-only summary (read_info):")
for s in qwen_info.samples:
    print(f"  [{s['id']}] n_tokens={s['n_tokens']}  scores={s.get('scores')}")


## Where to go next

- **Scale up.** Bump `LIMIT` (or pull `LEVEL="2"` / `"3"`) and increase `MAX_STEPS` / `MAX_TOKENS` for full GAIA runs. Together's Kimi K2.6 supports ≥260k context, so the per-turn cap is the practical ceiling.
- **Tag what matters.** Use `Sample.tag_by_text_regex` to tag specific tool names (`<tool_call name="web_search"`), specific URLs in the search-result blobs, or the agent's `submit` boundary for downstream attribution.
- **Probe positions.** With `Sample.materialize_position` you can attach `logit_lens` / `probe` results to specific tool-call openers (or the post-tool-result token where the agent decides whether to stop or keep searching) without re-tokenising the whole trace.
- **Compare models.** Re-run with `together/moonshotai/Kimi-K2.5` (or another tool-using model on Together) and use `InifDocument.subset` plus `Sample.sample_hash` to align samples across runs.